In [ ]:
from pathlib import Path
import pickle
import numpy as np
from glob import glob

# --- configure these ---
INPUT_DIR   = "raw_data"      # folder containing .pkl files (searched recursively)
OUTPUT_DIR  = "clean_data"    # where to save processed .pkl files
STATES_KEY  = "states"                 # the key that holds LiDAR arrays

MIN_RANGE   = 0.1                      # invalid if <= MIN_RANGE
MAX_RANGE   = 10.0                     # invalid if > MAX_RANGE

POLICY      = "nan"                    # one of: "nan", "clip", "const"
CONST_VALUE = 0.0                      # used only if POLICY == "const"

WRITE_MASK  = False                     # also write "<states_key>_valid_mask" (0/1)
OVERWRITE   = False                    # overwrite existing files in OUTPUT_DIR?
# ------------------------

INPUT_DIR  = Path(INPUT_DIR).expanduser().resolve()
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

Input:  C:\RewardFunction\raw_data
Output: C:\RewardFunction\clean_data


In [5]:
def clean_states(x, min_range=MIN_RANGE, max_range=MAX_RANGE, policy=POLICY, const_value=CONST_VALUE):
    """
    Applies threshold rules to a LiDAR array of any shape.
    Invalid if <= min_range, > max_range, or non-finite.
    Returns (cleaned_float32, valid_mask_uint8).
    """
    x = np.asarray(x, dtype=np.float32)
    valid = (x > min_range) & (x <= max_range) & np.isfinite(x)
    mask = valid.astype(np.uint8)

    if policy == "nan":
        y = x.copy()
        y[~valid] = np.nan
    elif policy == "clip":
        y = np.clip(x, min_range, max_range)
        y[~np.isfinite(x)] = np.nan  # keep original NaNs as NaN
    elif policy == "const":
        y = x.copy()
        y[~valid] = np.float32(const_value)
    else:
        raise ValueError("POLICY must be 'nan', 'clip', or 'const'.")

    return y, mask


def save_pickle(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


In [6]:
pkls = [Path(p) for p in glob(str(INPUT_DIR / "**" / "*.pkl"), recursive=True)]
if not pkls:
    raise SystemExit(f"No .pkl files found under {INPUT_DIR}")

written = 0
skipped = 0
failed  = 0

for in_path in pkls:
    # Determine output path mirroring the input folder structure
    rel = in_path.relative_to(INPUT_DIR)
    out_path = OUTPUT_DIR / rel

    if out_path.exists() and not OVERWRITE:
        print(f"[SKIP] {out_path} exists (set OVERWRITE=True to replace)")
        skipped += 1
        continue

    try:
        with in_path.open("rb") as f:
            obj = pickle.load(f)
    except Exception as e:
        print(f"[WARN] Failed to load {in_path}: {e}")
        failed += 1
        continue

    if not isinstance(obj, dict):
        print(f"[WARN] {in_path}: expected a dict with key '{STATES_KEY}'")
        failed += 1
        continue
    if STATES_KEY not in obj:
        print(f"[WARN] {in_path}: missing key '{STATES_KEY}'")
        failed += 1
        continue

    try:
        cleaned, mask = clean_states(obj[STATES_KEY])
        obj[STATES_KEY] = cleaned
        if WRITE_MASK:
            obj[f"{STATES_KEY}_valid_mask"] = mask
    except Exception as e:
        print(f"[WARN] {in_path}: cleaning failed: {e}")
        failed += 1
        continue

    try:
        save_pickle(obj, out_path)
        print(f"[OK]  {in_path} -> {out_path}")
        written += 1
    except Exception as e:
        print(f"[WARN] Failed to write {out_path}: {e}")
        failed += 1

print(f"\nDone. Wrote: {written} | Skipped: {skipped} | Failed: {failed}")

[OK]  C:\RewardFunction\raw_data\ftg_10_0.pkl -> C:\RewardFunction\clean_data\ftg_10_0.pkl
[OK]  C:\RewardFunction\raw_data\ftg_1_0.pkl -> C:\RewardFunction\clean_data\ftg_1_0.pkl
[OK]  C:\RewardFunction\raw_data\ftg_2_0.pkl -> C:\RewardFunction\clean_data\ftg_2_0.pkl
[OK]  C:\RewardFunction\raw_data\ftg_3_0.pkl -> C:\RewardFunction\clean_data\ftg_3_0.pkl
[OK]  C:\RewardFunction\raw_data\ftg_4_0.pkl -> C:\RewardFunction\clean_data\ftg_4_0.pkl
[OK]  C:\RewardFunction\raw_data\ftg_5_0.pkl -> C:\RewardFunction\clean_data\ftg_5_0.pkl
[OK]  C:\RewardFunction\raw_data\ftg_6_0.pkl -> C:\RewardFunction\clean_data\ftg_6_0.pkl
[OK]  C:\RewardFunction\raw_data\ftg_7_0.pkl -> C:\RewardFunction\clean_data\ftg_7_0.pkl
[OK]  C:\RewardFunction\raw_data\ftg_8_0.pkl -> C:\RewardFunction\clean_data\ftg_8_0.pkl
[OK]  C:\RewardFunction\raw_data\ftg_9_0.pkl -> C:\RewardFunction\clean_data\ftg_9_0.pkl
[OK]  C:\RewardFunction\raw_data\manual_drive_2_0.pkl -> C:\RewardFunction\clean_data\manual_drive_2_0.pkl
[